# 📘 Delta Lake Internals
## OPTIMIZE, VACUUM, Time Travel, Cloning & Schema Evolution

**What you'll learn:**
- How Delta Lake works under the hood (transaction log, file management)
- OPTIMIZE: compacting small files for better read performance
- VACUUM: cleaning up old files to reduce storage costs
- Time Travel: querying historical versions of your data
- Deep Clone vs Shallow Clone: when and why to use each
- Schema Evolution: adding, renaming, and changing column types safely
- MERGE patterns: 10+ different MERGE use cases
- Table properties and configurations
- Managed vs External vs Foreign tables
- RESTORE: recovering from mistakes
- Liquid Clustering internals

**Prerequisites:** Notebook 04 (Medallion Architecture — Delta Lake basics)
**Estimated Time:** 6-8 hours
**Database:** techmart_dw (existing from notebook 01)

---

> 📌 **Notebook 04** teaches you HOW to use Delta Lake (CREATE, INSERT, MERGE).
> **This notebook** teaches you how Delta Lake WORKS internally and how to OPTIMIZE it.

---

---
# 📗 Section 1: How Delta Lake Works Under the Hood

## The Transaction Log (_delta_log)

Every Delta table has a hidden `_delta_log/` directory that tracks ALL changes.
This is what gives Delta Lake its superpowers (ACID, time travel, audit history).

```
my_table/
├── _delta_log/                    ← Transaction log (the brain)
│   ├── 00000000000000000000.json  ← Version 0: table created
│   ├── 00000000000000000001.json  ← Version 1: first INSERT
│   ├── 00000000000000000002.json  ← Version 2: UPDATE some rows
│   ├── 00000000000000000003.json  ← Version 3: DELETE some rows
│   └── 00000000000000000010.checkpoint.parquet  ← Checkpoint (every 10 versions)
├── part-00000-xxx.parquet         ← Data file (actual data)
├── part-00001-xxx.parquet         ← Data file
├── part-00002-xxx.parquet         ← Data file (added in version 1)
└── part-00003-xxx.parquet         ← Data file (added in version 2)
```

### What's in a Transaction Log Entry?

Each JSON file in `_delta_log/` contains:
- **add**: Files added in this version
- **remove**: Files logically removed (still physically present until VACUUM)
- **metadata**: Schema changes, table properties
- **commitInfo**: Who made the change, when, what operation

### Key Insight: Copy-on-Write

Delta Lake NEVER modifies existing Parquet files. Instead:
- **INSERT**: Writes NEW Parquet files, logs them as "add"
- **UPDATE**: Writes NEW files with updated rows, marks old files as "remove"
- **DELETE**: Writes NEW files without deleted rows, marks old files as "remove"

This is why:
- Time travel works (old files still exist)
- VACUUM is needed (to actually delete old files)
- Concurrent reads are safe (readers see a consistent snapshot)

```
UPDATE orders SET status = 'shipped' WHERE order_id = 42

Before:                              After:
file_A.parquet (has order 42)        file_A.parquet (still exists! marked "remove")
file_B.parquet                       file_B.parquet (unchanged)
                                     file_C.parquet (NEW — has updated order 42)

_delta_log/version_N.json:
  remove: [file_A.parquet]
  add: [file_C.parquet]
```

In [ ]:
# Explore the Delta transaction log
# First, let's create a table and examine its internals

spark.sql("USE techmart_dw")

# Create a demo table for this notebook
spark.sql("DROP TABLE IF EXISTS techmart_dw.delta_internals_demo")
spark.sql("""
CREATE TABLE techmart_dw.delta_internals_demo (
    id INT,
    name STRING,
    amount DECIMAL(10,2),
    status STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

# Insert initial data
spark.sql("""
INSERT INTO techmart_dw.delta_internals_demo VALUES
(1, 'Alice', 100.00, 'active', current_timestamp()),
(2, 'Bob', 200.00, 'active', current_timestamp()),
(3, 'Charlie', 300.00, 'active', current_timestamp()),
(4, 'Diana', 400.00, 'inactive', current_timestamp()),
(5, 'Eve', 500.00, 'active', current_timestamp())
""")

print("✅ Created delta_internals_demo table with 5 rows")
print(f"   Row count: {spark.table('techmart_dw.delta_internals_demo').count()}")

In [ ]:
# View the table history (transaction log summary)
print("📜 TABLE HISTORY (DESCRIBE HISTORY)")
print("=" * 60)
spark.sql("DESCRIBE HISTORY techmart_dw.delta_internals_demo").show(truncate=False)

In [ ]:
# Make several modifications to build up history
# Version 2: UPDATE
spark.sql("""
UPDATE techmart_dw.delta_internals_demo 
SET amount = 150.00, updated_at = current_timestamp()
WHERE id = 1
""")
print("✅ Version 2: Updated Alice's amount to 150")

# Version 3: INSERT more rows
spark.sql("""
INSERT INTO techmart_dw.delta_internals_demo VALUES
(6, 'Frank', 600.00, 'active', current_timestamp()),
(7, 'Grace', 700.00, 'active', current_timestamp())
""")
print("✅ Version 3: Inserted Frank and Grace")

# Version 4: DELETE
spark.sql("""
DELETE FROM techmart_dw.delta_internals_demo WHERE id = 4
""")
print("✅ Version 4: Deleted Diana (id=4)")

# Version 5: MERGE (upsert)
spark.sql("""
MERGE INTO techmart_dw.delta_internals_demo AS target
USING (SELECT 2 AS id, 'Bob' AS name, 250.00 AS amount, 'premium' AS status, current_timestamp() AS updated_at
       UNION ALL
       SELECT 8, 'Henry', 800.00, 'active', current_timestamp()) AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
print("✅ Version 5: MERGE — updated Bob, inserted Henry")

# Show full history
print("\n📜 FULL TABLE HISTORY:")
spark.sql("DESCRIBE HISTORY techmart_dw.delta_internals_demo").show(truncate=False)

In [ ]:
# Examine table detail (file count, size)
print("📊 TABLE DETAIL")
print("=" * 60)
spark.sql("DESCRIBE DETAIL techmart_dw.delta_internals_demo").show(vertical=True)

---
# 📗 Section 2: Time Travel

## Querying Historical Versions

Delta Lake keeps old file versions, allowing you to query data AS OF any previous version.

### Syntax Options

```sql
-- By version number
SELECT * FROM my_table VERSION AS OF 3;

-- By timestamp
SELECT * FROM my_table TIMESTAMP AS OF '2024-03-15 10:00:00';

-- In Python/PySpark
df = spark.read.format("delta").option("versionAsOf", 3).load(path)
df = spark.read.format("delta").option("timestampAsOf", "2024-03-15").load(path)
```

### Use Cases for Time Travel

| Use Case | Example |
|----------|---------|
| **Audit** | "What did the data look like last Tuesday?" |
| **Debugging** | "When did this row get corrupted?" |
| **Rollback** | "Undo the last bad deployment" |
| **Comparison** | "How much did revenue change since yesterday?" |
| **Reproducibility** | "Train ML model on exact same data as last week" |
| **Recovery** | "Accidentally deleted rows — get them back" |

### Important: Time Travel Has Limits

- Old files are only available until VACUUM removes them
- Default retention: 7 days (configurable)
- After VACUUM, old versions are gone forever
- Checkpoints every 10 versions (for faster reads)

In [ ]:
# Time Travel — query different versions
print("⏰ TIME TRAVEL DEMO")
print("=" * 60)

# Current version
current = spark.table("techmart_dw.delta_internals_demo")
print(f"\n📌 CURRENT VERSION (latest):")
print(f"   Rows: {current.count()}")
current.show()

# Version 0 (just created, empty or initial)
print("\n📌 VERSION 1 (after initial insert):")
v1 = spark.sql("SELECT * FROM techmart_dw.delta_internals_demo VERSION AS OF 1")
print(f"   Rows: {v1.count()}")
v1.show()

# Version 2 (after UPDATE)
print("\n📌 VERSION 2 (after Alice's amount updated):")
v2 = spark.sql("SELECT * FROM techmart_dw.delta_internals_demo VERSION AS OF 2")
print(f"   Rows: {v2.count()}")
v2.orderBy("id").show()

In [ ]:
# Compare versions — what changed?
print("🔍 COMPARING VERSIONS (what changed between v1 and current?)")
print("=" * 60)

v1_df = spark.sql("SELECT * FROM techmart_dw.delta_internals_demo VERSION AS OF 1")
current_df = spark.table("techmart_dw.delta_internals_demo")

# Find new rows (in current but not in v1)
new_rows = current_df.join(v1_df, "id", "left_anti")
print(f"\n   New rows (added since v1): {new_rows.count()}")
new_rows.show()

# Find deleted rows (in v1 but not in current)
deleted_rows = v1_df.join(current_df, "id", "left_anti")
print(f"   Deleted rows (removed since v1): {deleted_rows.count()}")
deleted_rows.show()

# Find modified rows (same id, different values)
modified = (v1_df.alias("old")
    .join(current_df.alias("new"), "id")
    .where("old.amount != new.amount OR old.status != new.status"))
print(f"   Modified rows: {modified.count()}")
modified.select("id", "old.amount", "new.amount", "old.status", "new.status").show()

---
# 📗 Section 3: RESTORE — Undoing Mistakes

## Rolling Back to a Previous Version

If you accidentally corrupt data, RESTORE lets you roll back:

```sql
-- Restore to a specific version
RESTORE TABLE my_table TO VERSION AS OF 3;

-- Restore to a specific timestamp
RESTORE TABLE my_table TO TIMESTAMP AS OF '2024-03-15 10:00:00';
```

### How RESTORE Works

RESTORE doesn't delete the current version — it creates a NEW version that
looks like the old one. So you can even undo a RESTORE!

```
Version 0: Created
Version 1: INSERT 5 rows
Version 2: UPDATE (good)
Version 3: DELETE (accidental! deleted important data)
Version 4: RESTORE TO VERSION 2  ← New version that looks like v2
Version 5: (you can continue working from here)
```

### When to Use RESTORE vs Time Travel

| Scenario | Use |
|----------|-----|
| "Show me what data looked like yesterday" | Time Travel (SELECT ... VERSION AS OF) |
| "Undo the last change permanently" | RESTORE |
| "Compare current vs previous" | Time Travel (join two versions) |
| "Roll back a bad deployment" | RESTORE |

In [ ]:
# RESTORE demo — undo a mistake
print("🔄 RESTORE DEMO — Undoing a Mistake")
print("=" * 60)

# Current state
print("\n📌 CURRENT STATE (after all modifications):")
spark.sql("SELECT * FROM techmart_dw.delta_internals_demo ORDER BY id").show()
current_count = spark.table("techmart_dw.delta_internals_demo").count()
print(f"   Rows: {current_count}")

# Simulate a mistake: accidentally delete everything!
spark.sql("DELETE FROM techmart_dw.delta_internals_demo WHERE amount > 100")
after_mistake = spark.table("techmart_dw.delta_internals_demo").count()
print(f"\n❌ MISTAKE! Deleted rows where amount > 100")
print(f"   Rows remaining: {after_mistake} (lost {current_count - after_mistake} rows!)")

# RESTORE to before the mistake
spark.sql("RESTORE TABLE techmart_dw.delta_internals_demo TO VERSION AS OF 5")
after_restore = spark.table("techmart_dw.delta_internals_demo").count()
print(f"\n✅ RESTORED to version 5 (before the mistake)")
print(f"   Rows after restore: {after_restore}")
spark.sql("SELECT * FROM techmart_dw.delta_internals_demo ORDER BY id").show()

# Show that history still has everything
print("\n📜 History shows the restore:")
spark.sql("""
    SELECT version, operation, operationParameters 
    FROM (DESCRIBE HISTORY techmart_dw.delta_internals_demo)
    ORDER BY version DESC
    LIMIT 5
""").show(truncate=False)

---
# 📗 Section 4: OPTIMIZE — Compacting Small Files

## The Small Files Problem

Every INSERT, UPDATE, DELETE, and MERGE creates new Parquet files.
Over time, you accumulate thousands of tiny files → slow reads.

```
BEFORE OPTIMIZE:                         AFTER OPTIMIZE:
├── part-001.parquet (2 MB)              ├── part-001.parquet (128 MB)
├── part-002.parquet (500 KB)            ├── part-002.parquet (128 MB)
├── part-003.parquet (1 MB)              └── part-003.parquet (95 MB)
├── part-004.parquet (3 MB)
├── part-005.parquet (100 KB)            3 files, ~350 MB total
├── part-006.parquet (2 MB)              (same data, fewer files!)
├── ... (500 more tiny files)
└── part-500.parquet (1 MB)

500 files, ~350 MB total
(lots of overhead per file!)
```

### Why Small Files Are Bad

| Problem | Impact |
|---------|--------|
| **File listing overhead** | S3 LIST is slow with many files |
| **Per-file overhead** | Each file has metadata, footer, stats |
| **Parallelism waste** | One task per file → many tiny tasks |
| **Poor compression** | Small files compress poorly |
| **Slow reads** | More I/O operations needed |

### OPTIMIZE Syntax

```sql
-- Optimize entire table
OPTIMIZE my_table;

-- Optimize specific partition (faster for large tables)
OPTIMIZE my_table WHERE date = '2024-03-15';

-- ZORDER (legacy — use Liquid Clustering for new tables)
OPTIMIZE my_table ZORDER BY (customer_id, order_date);
```

### When to Run OPTIMIZE

| Scenario | Frequency |
|----------|-----------|
| After many small INSERTs | After each batch (or use autoCompact) |
| After streaming writes | Every 1-4 hours |
| Before heavy read workloads | Before dashboard refresh |
| After MERGE operations | After each MERGE |

> 💡 **2025 Best Practice:** Enable `autoCompact` and `optimizeWrite` to reduce
> the need for manual OPTIMIZE. Use Predictive Optimization for automatic scheduling.

In [ ]:
# OPTIMIZE demo
print("🗜️ OPTIMIZE DEMO")
print("=" * 60)

# Create a table with many small files (simulate streaming writes)
spark.sql("DROP TABLE IF EXISTS techmart_dw.small_files_demo")
spark.sql("""
CREATE TABLE techmart_dw.small_files_demo (
    id INT, category STRING, amount DECIMAL(10,2), event_date DATE
) USING DELTA
""")

# Insert data in many small batches (creates many small files)
for i in range(10):
    spark.sql(f"""
        INSERT INTO techmart_dw.small_files_demo
        SELECT 
            id + {i * 100},
            CASE (id + {i}) % 5 
                WHEN 0 THEN 'electronics' WHEN 1 THEN 'clothing'
                WHEN 2 THEN 'food' WHEN 3 THEN 'books' ELSE 'other' END,
            CAST(abs(hash(id * 7 + {i})) % 1000 + 1 AS DECIMAL(10,2)),
            date_add('2024-01-01', abs(hash(id * 13 + {i})) % 365)
        FROM (SELECT explode(sequence(1, 100)) AS id)
    """)

print("Created table with 10 small inserts (1000 rows total)")

# Check file count BEFORE optimize
detail_before = spark.sql("DESCRIBE DETAIL techmart_dw.small_files_demo").collect()[0]
print(f"\n📊 BEFORE OPTIMIZE:")
print(f"   Number of files: {detail_before['numFiles']}")
print(f"   Size: {detail_before['sizeInBytes'] / 1024:.1f} KB")
print(f"   Avg file size: {detail_before['sizeInBytes'] / max(detail_before['numFiles'], 1) / 1024:.1f} KB")

# Run OPTIMIZE
print("\n🔄 Running OPTIMIZE...")
result = spark.sql("OPTIMIZE techmart_dw.small_files_demo")
result.show(truncate=False)

# Check file count AFTER optimize
detail_after = spark.sql("DESCRIBE DETAIL techmart_dw.small_files_demo").collect()[0]
print(f"\n📊 AFTER OPTIMIZE:")
print(f"   Number of files: {detail_after['numFiles']}")
print(f"   Size: {detail_after['sizeInBytes'] / 1024:.1f} KB")
print(f"   Avg file size: {detail_after['sizeInBytes'] / max(detail_after['numFiles'], 1) / 1024:.1f} KB")

reduction = (1 - detail_after['numFiles'] / max(detail_before['numFiles'], 1)) * 100
print(f"\n   📉 File count reduced by {reduction:.0f}%")

---
# 📗 Section 5: VACUUM — Cleaning Up Old Files

## Why VACUUM is Needed

Remember: Delta Lake never modifies files. UPDATE/DELETE create NEW files and
mark old ones as "removed" in the transaction log. But the old files still exist on disk!

```
After UPDATE (without VACUUM):
├── file_A.parquet  ← STILL ON DISK (marked "remove" in log)
├── file_B.parquet  ← Current (active)
├── file_C.parquet  ← Current (active, has updated rows)

After VACUUM:
├── file_B.parquet  ← Current
├── file_C.parquet  ← Current
(file_A is physically deleted — saves storage!)
```

### VACUUM Syntax

```sql
-- Remove files older than 7 days (default retention)
VACUUM my_table;

-- Remove files older than 24 hours (minimum allowed)
VACUUM my_table RETAIN 24 HOURS;

-- Dry run (show what WOULD be deleted, don't actually delete)
VACUUM my_table DRY RUN;
```

### VACUUM Rules

| Rule | Details |
|------|---------|
| **Default retention** | 7 days (168 hours) |
| **Minimum retention** | 24 hours (safety check) |
| **Override minimum** | `spark.databricks.delta.retentionDurationCheck.enabled = false` |
| **What it deletes** | Files not referenced by any version within retention |
| **What it preserves** | Files needed for current version + retention window |
| **Impact on time travel** | Versions older than retention become inaccessible |

### ⚠️ VACUUM Destroys Time Travel!

```
Retention = 7 days

Day 1: Version 0 (CREATE)
Day 2: Version 1 (INSERT)
Day 3: Version 2 (UPDATE)
Day 8: VACUUM runs
        → Versions 0 and 1 files may be deleted
        → Time travel to version 0 or 1 will FAIL
        → Version 2+ still works
```

### Best Practices

1. **Don't set retention too low** — 7 days is a good default
2. **Run VACUUM regularly** — weekly or let Predictive Optimization handle it
3. **Never VACUUM with 0 HOURS** — you'll break concurrent readers
4. **Use DRY RUN first** — see what will be deleted before committing

In [ ]:
# VACUUM demo
print("🧹 VACUUM DEMO")
print("=" * 60)

# Check current table size (includes old files)
detail = spark.sql("DESCRIBE DETAIL techmart_dw.small_files_demo").collect()[0]
print(f"\n📊 Current table state:")
print(f"   Size on disk: {detail['sizeInBytes'] / 1024:.1f} KB")
print(f"   Active files: {detail['numFiles']}")

# Show history (VACUUM will affect time travel for old versions)
print("\n📜 Table history:")
spark.sql("""
    SELECT version, operation, timestamp
    FROM (DESCRIBE HISTORY techmart_dw.small_files_demo)
    ORDER BY version
""").show(truncate=False)

# DRY RUN — see what would be deleted
print("\n🔍 VACUUM DRY RUN (what would be deleted):")
# Note: On Community Edition, files may already be cleaned up
# In production, this shows files that are safe to delete
try:
    spark.sql("VACUUM techmart_dw.small_files_demo RETAIN 0 HOURS DRY RUN")
except Exception as e:
    print(f"   Expected: {e}")
    print("   (Safety check prevents VACUUM with 0 hours retention)")
    print("   Use: VACUUM table RETAIN 168 HOURS (7 days, default)")

# Run actual VACUUM with default retention
print("\n🧹 Running VACUUM with default retention (7 days):")
spark.sql("VACUUM techmart_dw.small_files_demo")
print("   ✅ VACUUM complete")

# Check size after
detail_after = spark.sql("DESCRIBE DETAIL techmart_dw.small_files_demo").collect()[0]
print(f"\n📊 After VACUUM:")
print(f"   Size on disk: {detail_after['sizeInBytes'] / 1024:.1f} KB")
print(f"   Active files: {detail_after['numFiles']}")

---
# 📗 Section 6: Deep Clone vs Shallow Clone

## Table Cloning

Cloning creates a copy of a Delta table. Two types:

### Deep Clone
- **Full physical copy** of all data files
- Independent of source (source can be deleted)
- Takes time and storage proportional to table size
- Use for: production backups, creating test environments

### Shallow Clone
- **Only copies metadata** (transaction log), not data files
- References source table's data files
- Fast and cheap (no data copied)
- Breaks if source files are VACUUMed
- Use for: quick testing, experimentation, short-lived copies

```
DEEP CLONE:                              SHALLOW CLONE:
┌──────────────┐  ┌──────────────┐      ┌──────────────┐  ┌──────────────┐
│   Source     │  │    Clone     │      │   Source     │  │    Clone     │
│              │  │              │      │              │  │  (metadata   │
│  data files  │  │  data files  │      │  data files ◄├──┤   only)      │
│  (copied!)   │  │  (full copy) │      │              │  │              │
└──────────────┘  └──────────────┘      └──────────────┘  └──────────────┘
Independent — source can be deleted      Dependent — needs source files!
```

### Syntax

```sql
-- Deep Clone (full copy)
CREATE TABLE my_backup DEEP CLONE my_source;

-- Shallow Clone (metadata only)
CREATE TABLE my_test SHALLOW CLONE my_source;

-- Clone with filtering
CREATE TABLE q1_backup DEEP CLONE my_source
WHERE order_date BETWEEN '2024-01-01' AND '2024-03-31';

-- Clone to a specific version (time travel + clone)
CREATE TABLE yesterday_backup DEEP CLONE my_source VERSION AS OF 5;
```

### When to Use Each

| Scenario | Clone Type | Why |
|----------|-----------|-----|
| Production backup | Deep | Must survive source changes |
| Create test environment | Shallow | Fast, temporary, cheap |
| Migrate table to new location | Deep | Need independent copy |
| Experiment with schema changes | Shallow | Quick, disposable |
| Disaster recovery | Deep | Must be independent |
| A/B testing data | Shallow | Short-lived comparison |

In [ ]:
# Cloning demo
print("📋 CLONING DEMO")
print("=" * 60)

# Source table
source_count = spark.table("techmart_dw.delta_internals_demo").count()
print(f"\n📌 Source table: delta_internals_demo ({source_count} rows)")

# Deep Clone
print("\n🔵 DEEP CLONE (full physical copy):")
spark.sql("DROP TABLE IF EXISTS techmart_dw.demo_deep_clone")
spark.sql("CREATE TABLE techmart_dw.demo_deep_clone DEEP CLONE techmart_dw.delta_internals_demo")
deep_count = spark.table("techmart_dw.demo_deep_clone").count()
print(f"   Created: demo_deep_clone ({deep_count} rows)")
print("   ✅ Independent copy — source can be modified/deleted safely")

# Shallow Clone
print("\n🟡 SHALLOW CLONE (metadata only):")
spark.sql("DROP TABLE IF EXISTS techmart_dw.demo_shallow_clone")
spark.sql("CREATE TABLE techmart_dw.demo_shallow_clone SHALLOW CLONE techmart_dw.delta_internals_demo")
shallow_count = spark.table("techmart_dw.demo_shallow_clone").count()
print(f"   Created: demo_shallow_clone ({shallow_count} rows)")
print("   ⚠️ References source files — breaks if source is VACUUMed")

# Modify the deep clone (doesn't affect source)
spark.sql("INSERT INTO techmart_dw.demo_deep_clone VALUES (99, 'Clone-Only', 999.99, 'test', current_timestamp())")
deep_after = spark.table("techmart_dw.demo_deep_clone").count()
source_after = spark.table("techmart_dw.delta_internals_demo").count()
print(f"\n📊 After modifying deep clone:")
print(f"   Deep clone rows: {deep_after} (added 1)")
print(f"   Source rows: {source_after} (unchanged!)")
print("   ✅ Deep clone is fully independent")

# Compare sizes
for table_name in ["delta_internals_demo", "demo_deep_clone", "demo_shallow_clone"]:
    detail = spark.sql(f"DESCRIBE DETAIL techmart_dw.{table_name}").collect()[0]
    print(f"\n   {table_name}: {detail['sizeInBytes']/1024:.1f} KB, {detail['numFiles']} files")

---
# 📗 Section 7: Schema Evolution

## Handling Schema Changes Safely

In production, schemas change: new columns are added, types change, columns are renamed.
Delta Lake provides tools to handle this gracefully.

### Schema Evolution Options

| Change Type | How to Handle | Risk Level |
|-------------|--------------|------------|
| **Add column** | `ALTER TABLE ADD COLUMN` or `mergeSchema` | Low |
| **Rename column** | `ALTER TABLE RENAME COLUMN` | Medium |
| **Change type** | `ALTER TABLE ALTER COLUMN` (widening only) | Medium |
| **Drop column** | `ALTER TABLE DROP COLUMN` | High |
| **Reorder columns** | Not directly supported | N/A |

### Safe Type Widening (Allowed)

```
byte → short → int → long → decimal
float → double
string stays string (can't narrow)
```

### mergeSchema Option

When writing data with new columns, use `mergeSchema` to auto-add them:

```python
# Auto-add new columns during write
df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("my_table")
```

### overwriteSchema Option

When you need to completely replace the schema:

```python
# Replace entire schema (dangerous!)
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_table")
```

In [ ]:
# Schema Evolution demo
print("🔄 SCHEMA EVOLUTION DEMO")
print("=" * 60)

# Show current schema
print("\n📌 Current schema:")
spark.sql("DESCRIBE techmart_dw.delta_internals_demo").show()

# 1. ADD COLUMN
print("\n1️⃣ ADD COLUMN (safe, non-breaking):")
spark.sql("ALTER TABLE techmart_dw.delta_internals_demo ADD COLUMN (email STRING)")
print("   Added 'email' column (existing rows will have NULL)")
spark.sql("SELECT id, name, email FROM techmart_dw.delta_internals_demo LIMIT 3").show()

# 2. ADD COLUMN with DEFAULT
print("\n2️⃣ ADD COLUMN with comment:")
spark.sql("ALTER TABLE techmart_dw.delta_internals_demo ADD COLUMN (tier STRING COMMENT 'Customer tier')")
print("   Added 'tier' column with comment")

# 3. RENAME COLUMN
print("\n3️⃣ RENAME COLUMN:")
spark.sql("ALTER TABLE techmart_dw.delta_internals_demo RENAME COLUMN tier TO customer_tier")
print("   Renamed 'tier' → 'customer_tier'")

# Show updated schema
print("\n📌 Updated schema:")
spark.sql("DESCRIBE techmart_dw.delta_internals_demo").show()

In [ ]:
# Schema evolution with mergeSchema (auto-add columns during write)
print("🔄 mergeSchema — Auto-add columns during write")
print("=" * 60)

# Create a DataFrame with a NEW column not in the table
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, TimestampType

new_data = spark.createDataFrame([
    (100, "NewPerson", 1000.00, "active", "2024-03-15 10:00:00", "new@email.com", "gold", "US")
], ["id", "name", "amount", "status", "updated_at", "email", "customer_tier", "country"])

# This would FAIL without mergeSchema (because 'country' doesn't exist in table)
print("   Writing data with new 'country' column...")
try:
    new_data.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("techmart_dw.delta_internals_demo")
    print("   ✅ Success! 'country' column auto-added to table schema")
except Exception as e:
    # On Community Edition, saveAsTable may not work for existing tables
    # Use SQL instead
    spark.sql("ALTER TABLE techmart_dw.delta_internals_demo ADD COLUMN (country STRING)")
    spark.sql("INSERT INTO techmart_dw.delta_internals_demo VALUES (100, 'NewPerson', 1000.00, 'active', current_timestamp(), 'new@email.com', 'gold', 'US')")
    print("   ✅ Added 'country' column and inserted row")

print("\n📌 Final schema:")
spark.sql("DESCRIBE techmart_dw.delta_internals_demo").show()

---
# 📗 Section 8: MERGE Patterns — 10 Use Cases

## The MERGE Statement

MERGE is Delta Lake's most powerful operation. It combines INSERT, UPDATE, and DELETE
in a single atomic transaction.

```sql
MERGE INTO target
USING source
ON target.id = source.id
WHEN MATCHED AND source.operation = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

### 10 MERGE Patterns

| # | Pattern | Use Case |
|---|---------|----------|
| 1 | **Simple Upsert** | Insert new, update existing |
| 2 | **Insert-Only** | Only insert new records (ignore existing) |
| 3 | **Update-Only** | Only update existing (ignore new) |
| 4 | **Soft Delete** | Mark deleted records (don't physically remove) |
| 5 | **Hard Delete** | Physically remove records |
| 6 | **SCD Type 1** | Overwrite with latest values |
| 7 | **SCD Type 2** | Keep history with versioning |
| 8 | **Conditional Update** | Update only specific columns |
| 9 | **Deduplication** | Remove duplicates on insert |
| 10 | **CDC Apply** | Apply Change Data Capture events |

In [ ]:
# Setup: Create tables for MERGE demos
spark.sql("DROP TABLE IF EXISTS techmart_dw.merge_target")
spark.sql("""
CREATE TABLE techmart_dw.merge_target (
    customer_id INT,
    name STRING,
    email STRING,
    status STRING,
    lifetime_value DECIMAL(12,2),
    updated_at TIMESTAMP
) USING DELTA
""")

spark.sql("""
INSERT INTO techmart_dw.merge_target VALUES
(1, 'Alice Smith', 'alice@example.com', 'active', 1500.00, '2024-01-01 00:00:00'),
(2, 'Bob Jones', 'bob@example.com', 'active', 2500.00, '2024-01-01 00:00:00'),
(3, 'Charlie Brown', 'charlie@example.com', 'inactive', 500.00, '2024-01-01 00:00:00'),
(4, 'Diana Prince', 'diana@example.com', 'active', 3000.00, '2024-01-01 00:00:00'),
(5, 'Eve Wilson', 'eve@example.com', 'active', 1000.00, '2024-01-01 00:00:00')
""")

print("✅ Created merge_target with 5 customers")
spark.sql("SELECT * FROM techmart_dw.merge_target ORDER BY customer_id").show(truncate=False)

In [ ]:
# Pattern 1: Simple Upsert (most common)
print("📌 PATTERN 1: Simple Upsert (Insert new, Update existing)")
print("=" * 60)

spark.sql("""
MERGE INTO techmart_dw.merge_target AS target
USING (
    SELECT 2 AS customer_id, 'Bob Jones' AS name, 'bob.new@example.com' AS email, 
           'premium' AS status, 3000.00 AS lifetime_value, current_timestamp() AS updated_at
    UNION ALL
    SELECT 6, 'Frank Miller', 'frank@example.com', 'active', 750.00, current_timestamp()
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("   Updated Bob (id=2), Inserted Frank (id=6)")
spark.sql("SELECT * FROM techmart_dw.merge_target WHERE customer_id IN (2, 6)").show(truncate=False)

In [ ]:
# Pattern 2: Insert-Only Merge (skip existing records)
print("📌 PATTERN 2: Insert-Only (ignore duplicates)")
print("=" * 60)

spark.sql("""
MERGE INTO techmart_dw.merge_target AS target
USING (
    SELECT 1 AS customer_id, 'Alice DUPLICATE' AS name, 'dup@example.com' AS email,
           'active' AS status, 9999.00 AS lifetime_value, current_timestamp() AS updated_at
    UNION ALL
    SELECT 7, 'Grace Hopper', 'grace@example.com', 'active', 2000.00, current_timestamp()
) AS source
ON target.customer_id = source.customer_id
WHEN NOT MATCHED THEN INSERT *
""")

print("   Alice (id=1) NOT updated (insert-only pattern)")
print("   Grace (id=7) inserted (new record)")
spark.sql("SELECT customer_id, name, lifetime_value FROM techmart_dw.merge_target WHERE customer_id IN (1, 7)").show()

In [ ]:
# Pattern 3: Soft Delete (mark as deleted, don't remove)
print("📌 PATTERN 3: Soft Delete")
print("=" * 60)

# First add a deleted_at column
spark.sql("ALTER TABLE techmart_dw.merge_target ADD COLUMN (deleted_at TIMESTAMP)")

spark.sql("""
MERGE INTO techmart_dw.merge_target AS target
USING (
    SELECT 3 AS customer_id  -- Charlie should be soft-deleted
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET 
    target.status = 'deleted',
    target.deleted_at = current_timestamp()
""")

print("   Charlie (id=3) soft-deleted (status='deleted', deleted_at set)")
spark.sql("SELECT customer_id, name, status, deleted_at FROM techmart_dw.merge_target WHERE customer_id = 3").show(truncate=False)

In [ ]:
# Pattern 4: CDC Apply (handle INSERT/UPDATE/DELETE operations)
print("📌 PATTERN 4: CDC Apply (Change Data Capture)")
print("=" * 60)

# Simulate CDC events
spark.sql("""
CREATE OR REPLACE TEMP VIEW cdc_events AS
SELECT * FROM VALUES
    ('U', 4, 'Diana Prince', 'diana.new@example.com', 'premium', 5000.00, current_timestamp(), NULL),
    ('D', 5, 'Eve Wilson', 'eve@example.com', 'deleted', 1000.00, current_timestamp(), NULL),
    ('I', 8, 'Henry Adams', 'henry@example.com', 'active', 1200.00, current_timestamp(), NULL)
AS cdc(operation, customer_id, name, email, status, lifetime_value, updated_at, deleted_at)
""")

spark.sql("""
MERGE INTO techmart_dw.merge_target AS target
USING cdc_events AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.operation = 'D' THEN DELETE
WHEN MATCHED AND source.operation = 'U' THEN UPDATE SET 
    target.name = source.name,
    target.email = source.email,
    target.status = source.status,
    target.lifetime_value = source.lifetime_value,
    target.updated_at = source.updated_at
WHEN NOT MATCHED AND source.operation = 'I' THEN INSERT 
    (customer_id, name, email, status, lifetime_value, updated_at)
    VALUES (source.customer_id, source.name, source.email, source.status, source.lifetime_value, source.updated_at)
""")

print("   Applied CDC: Updated Diana, Deleted Eve, Inserted Henry")
spark.sql("SELECT customer_id, name, status, lifetime_value FROM techmart_dw.merge_target ORDER BY customer_id").show()

---
# 📗 Section 9: Table Properties & Configuration

## Key Delta Table Properties

| Property | What It Does | Default |
|----------|-------------|---------|
| `delta.autoOptimize.optimizeWrite` | Coalesce small files on write | true (DBR) |
| `delta.autoOptimize.autoCompact` | Auto-compact after writes | true (DBR) |
| `delta.logRetentionDuration` | How long to keep transaction log | 30 days |
| `delta.deletedFileRetentionDuration` | How long before VACUUM can delete | 7 days |
| `delta.enableChangeDataFeed` | Enable Change Data Feed (CDF) | false |
| `delta.columnMapping.mode` | Enable column rename/drop | 'name' |
| `delta.minReaderVersion` | Minimum reader protocol version | varies |
| `delta.minWriterVersion` | Minimum writer protocol version | varies |

## Table Types

| Type | Data Location | DROP Behavior | Use Case |
|------|--------------|---------------|----------|
| **Managed** | Controlled by catalog | DROP deletes data + metadata | Default, most tables |
| **External** | You specify location | DROP deletes metadata only | Shared storage, multiple engines |
| **Temporary** | Session memory | Gone when session ends | Intermediate processing |
| **View** | No data (just SQL) | DROP removes definition | Abstraction layer |

In [ ]:
# Table properties demo
print("⚙️ TABLE PROPERTIES")
print("=" * 60)

# Show current properties
print("\n📌 Current table properties:")
spark.sql("SHOW TBLPROPERTIES techmart_dw.delta_internals_demo").show(truncate=False)

# Set useful properties
spark.sql("""
ALTER TABLE techmart_dw.delta_internals_demo SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.logRetentionDuration' = 'interval 30 days',
    'delta.deletedFileRetentionDuration' = 'interval 7 days'
)
""")
print("\n✅ Set optimization properties")

# Enable Change Data Feed
spark.sql("""
ALTER TABLE techmart_dw.delta_internals_demo SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")
print("✅ Enabled Change Data Feed (CDF)")

# Show updated properties
print("\n📌 Updated properties:")
spark.sql("SHOW TBLPROPERTIES techmart_dw.delta_internals_demo").show(truncate=False)

---
# 📗 Section 10: Change Data Feed (CDF)

## What is CDF?

Change Data Feed records ROW-LEVEL changes to a Delta table.
After enabling CDF, you can query what changed between versions.

```sql
-- Query changes between versions
SELECT * FROM table_changes('my_table', 2, 5);

-- Query changes by timestamp
SELECT * FROM table_changes('my_table', '2024-03-15', '2024-03-16');
```

### CDF Output Columns

| Column | Description |
|--------|-------------|
| `_change_type` | 'insert', 'update_preimage', 'update_postimage', 'delete' |
| `_commit_version` | Version number of the change |
| `_commit_timestamp` | Timestamp of the change |

### Use Cases

- **Incremental ETL**: Process only changed rows (not full table)
- **Audit trail**: Track who changed what, when
- **CDC downstream**: Feed changes to downstream systems
- **Data sync**: Keep replica tables in sync

In [ ]:
# Change Data Feed demo
print("📡 CHANGE DATA FEED (CDF) DEMO")
print("=" * 60)

# CDF was enabled above. Now make some changes and query them.
# Get current version
current_version = spark.sql("""
    SELECT MAX(version) as v FROM (DESCRIBE HISTORY techmart_dw.delta_internals_demo)
""").collect()[0]["v"]
print(f"   Current version: {current_version}")

# Make a change
spark.sql("""
UPDATE techmart_dw.delta_internals_demo 
SET status = 'vip', updated_at = current_timestamp()
WHERE customer_id = 1
""")
print("   Updated customer 1 to 'vip' status")

# Insert a new record
spark.sql("""
INSERT INTO techmart_dw.delta_internals_demo (customer_id, name, amount, status, updated_at)
VALUES (200, 'CDF Test User', 999.99, 'new', current_timestamp())
""")
print("   Inserted new customer (id=200)")

# Query the changes using CDF
new_version = current_version + 2
print(f"\n📡 Changes from version {current_version + 1} to {new_version}:")
try:
    changes = spark.sql(f"""
        SELECT customer_id, name, status, _change_type, _commit_version
        FROM table_changes('techmart_dw.delta_internals_demo', {current_version + 1}, {new_version})
        ORDER BY _commit_version, _change_type
    """)
    changes.show(truncate=False)
    
    print("   Change types explained:")
    print("   • update_preimage = old values (before update)")
    print("   • update_postimage = new values (after update)")
    print("   • insert = new row added")
except Exception as e:
    print(f"   ⚠️ CDF query failed: {e}")
    print("   (CDF may need a version after enablement to work)")

---
# 📗 Section 11: Practice Exercises

## Exercise 1: Time Travel Investigation

**Scenario:** A report shows incorrect revenue numbers for yesterday.
You need to find when the data was corrupted.

In [ ]:
# ============================================
# 🎯 YOUR TURN: Time Travel Investigation
# ============================================
# The merge_target table has been modified multiple times.
# 
# Tasks:
# 1. Use DESCRIBE HISTORY to see all versions
# 2. Query version 0 (original data) — what was Diana's lifetime_value?
# 3. Query the current version — what is it now?
# 4. Find which version changed Diana's data
#
# Write your investigation below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
print("🔍 TIME TRAVEL INVESTIGATION")
print("=" * 60)

# 1. Show history
print("\n1️⃣ Table history:")
spark.sql("DESCRIBE HISTORY techmart_dw.merge_target").show(truncate=50)

# 2. Original value
print("\n2️⃣ Diana's original lifetime_value (version 0):")
spark.sql("""
    SELECT customer_id, name, lifetime_value, status
    FROM techmart_dw.merge_target VERSION AS OF 0
    WHERE customer_id = 4
""").show()

# 3. Current value
print("\n3️⃣ Diana's current lifetime_value:")
spark.sql("""
    SELECT customer_id, name, lifetime_value, status
    FROM techmart_dw.merge_target
    WHERE customer_id = 4
""").show()

print("\n4️⃣ The change happened during the CDC MERGE (Pattern 4)")
print("   Diana was updated from $3000 → $5000 and status changed to 'premium'")

## Exercise 2: OPTIMIZE and VACUUM Strategy

In [ ]:
# ============================================
# 🎯 YOUR TURN: Design an OPTIMIZE/VACUUM Strategy
# ============================================
# Scenario: You have a table that receives:
# - 50 small INSERT batches per day (streaming micro-batches)
# - 1 large MERGE per night (CDC from source system)
# - Analysts query it 100+ times per day
# - Compliance requires 30-day data history
#
# Design:
# 1. How often should you OPTIMIZE?
# 2. What VACUUM retention should you set?
# 3. Should you enable autoCompact? optimizeWrite?
# 4. What Liquid Clustering keys would you choose?
#
# Write your strategy below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
optimization_strategy = {
    "OPTIMIZE frequency": {
        "recommendation": "Every 4 hours during business hours + after nightly MERGE",
        "reason": "50 micro-batches/day create small files. 4-hour cadence keeps reads fast.",
        "alternative": "Enable autoCompact=true to handle this automatically"
    },
    "VACUUM retention": {
        "recommendation": "RETAIN 720 HOURS (30 days)",
        "reason": "Compliance requires 30-day history. VACUUM after 30 days.",
        "config": "delta.deletedFileRetentionDuration = 'interval 30 days'"
    },
    "autoCompact": {
        "recommendation": "ENABLED",
        "reason": "Automatically compacts small files after each write. "
                  "Reduces need for manual OPTIMIZE."
    },
    "optimizeWrite": {
        "recommendation": "ENABLED",
        "reason": "Coalesces small partitions during write. "
                  "Fewer small files created in the first place."
    },
    "Liquid Clustering": {
        "recommendation": "CLUSTER BY (event_date, customer_id)",
        "reason": "Analysts filter by date (partition pruning) and join on customer_id. "
                  "These are the most common access patterns."
    },
    "Predictive Optimization": {
        "recommendation": "ENABLED (if available)",
        "reason": "Let Databricks AI determine optimal OPTIMIZE/VACUUM schedule. "
                  "Eliminates manual scheduling entirely."
    }
}

print("📋 OPTIMIZATION STRATEGY")
print("=" * 60)
for component, details in optimization_strategy.items():
    print(f"\n📌 {component}:")
    print(f"   Recommendation: {details['recommendation']}")
    print(f"   Reason: {details['reason']}")
    if 'config' in details:
        print(f"   Config: {details['config']}")


---
# 📗 Section 12: Summary & Quick Reference

## Delta Lake Internals Cheat Sheet

```
TRANSACTION LOG:
  _delta_log/ contains JSON files (one per version)
  Each file records: adds, removes, metadata changes
  Checkpoints every 10 versions (Parquet format, faster reads)

TIME TRAVEL:
  SELECT * FROM table VERSION AS OF 3;
  SELECT * FROM table TIMESTAMP AS OF '2024-03-15';
  Works until VACUUM removes old files

RESTORE:
  RESTORE TABLE my_table TO VERSION AS OF 3;
  Creates a NEW version that looks like the old one
  Can undo a RESTORE (it's just another version)

OPTIMIZE:
  OPTIMIZE my_table;                    -- Compact all files
  OPTIMIZE my_table WHERE date = '...'; -- Compact one partition
  Target file size: 128 MB (configurable)
  Enable autoCompact for automatic compaction

VACUUM:
  VACUUM my_table;                      -- Default 7-day retention
  VACUUM my_table RETAIN 720 HOURS;     -- 30-day retention
  VACUUM my_table DRY RUN;              -- Preview only
  ⚠️ Destroys time travel for versions beyond retention!

CLONING:
  CREATE TABLE backup DEEP CLONE source;    -- Full copy (independent)
  CREATE TABLE test SHALLOW CLONE source;   -- Metadata only (dependent)

SCHEMA EVOLUTION:
  ALTER TABLE t ADD COLUMN (col TYPE);      -- Safe
  ALTER TABLE t RENAME COLUMN old TO new;   -- Medium risk
  ALTER TABLE t ALTER COLUMN col TYPE new;  -- Widening only
  option("mergeSchema", "true")             -- Auto-add on write

MERGE (top patterns):
  Upsert: WHEN MATCHED UPDATE, WHEN NOT MATCHED INSERT
  Insert-only: WHEN NOT MATCHED INSERT (skip existing)
  CDC: WHEN MATCHED AND op='D' DELETE, WHEN MATCHED UPDATE, WHEN NOT MATCHED INSERT

CHANGE DATA FEED:
  ALTER TABLE t SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
  SELECT * FROM table_changes('my_table', start_version, end_version);

TABLE PROPERTIES:
  delta.autoOptimize.optimizeWrite = true
  delta.autoOptimize.autoCompact = true
  delta.logRetentionDuration = 'interval 30 days'
  delta.deletedFileRetentionDuration = 'interval 7 days'
  delta.enableChangeDataFeed = true
```

## Next Steps

- **Notebook 04:** Medallion Architecture (Delta Lake usage patterns)
- **Notebook 07:** Performance Optimization (Spark tuning)
- **Notebook 48:** CDC Change Data Capture (using CDF in pipelines)
- **Notebook 33:** Platform Essentials (Liquid Clustering, Predictive Optimization)

---
*Notebook 46 of the Databricks Data Engineering series*
*Study Order: 46 (Advanced Topics)*

---
# 📗 Section 13: Delta Lake Internals Summary

## How Delta Lake Works -- The Complete Picture

```
EVERY WRITE creates new Parquet files:
  INSERT -> new files added (logged as "add")
  UPDATE -> new files with updated rows (old files logged as "remove")
  DELETE -> new files without deleted rows (old files logged as "remove")

_delta_log/ tracks everything:
  00000000000000000000.json -> Version 0 (CREATE TABLE)
  00000000000000000001.json -> Version 1 (first INSERT)
  00000000000000000002.json -> Version 2 (UPDATE)
  00000000000000000010.checkpoint.parquet -> Checkpoint (every 10 versions)

TIME TRAVEL reads old versions:
  SELECT * FROM table VERSION AS OF 3
  -> Reads files that were "active" at version 3

VACUUM removes old files:
  VACUUM table RETAIN 168 HOURS
  -> Deletes files marked "remove" older than 7 days
  -> After VACUUM, time travel to those versions is impossible!
```

## Key Commands Reference

| Command | Purpose | Example |
|---------|---------|---------|
| `DESCRIBE HISTORY` | Show all versions | `DESCRIBE HISTORY orders` |
| `DESCRIBE DETAIL` | Show file count, size | `DESCRIBE DETAIL orders` |
| `OPTIMIZE` | Compact small files | `OPTIMIZE orders` |
| `VACUUM` | Delete old files | `VACUUM orders RETAIN 168 HOURS` |
| `RESTORE` | Roll back to version | `RESTORE TABLE orders TO VERSION AS OF 5` |
| `CLONE` | Copy table | `CREATE TABLE backup DEEP CLONE orders` |
| `ALTER TABLE ... CLUSTER BY` | Set Liquid Clustering | `ALTER TABLE orders CLUSTER BY (date, region)` |

## Performance Optimization Checklist

- [ ] Enable `optimizeWrite` and `autoCompact`
- [ ] Use Liquid Clustering on filter columns
- [ ] Enable Predictive Optimization (auto OPTIMIZE/VACUUM)
- [ ] Set appropriate retention (7 days default)
- [ ] Enable Change Data Feed if downstream needs it
- [ ] Run ANALYZE TABLE to update statistics

In [ ]:
%sql
-- Delta Lake internals commands
-- Run these on any Delta table to inspect its state

-- 1. Show table history
DESCRIBE HISTORY techmart_dw.orders;

-- 2. Show table details (file count, size)
DESCRIBE DETAIL techmart_dw.orders;

-- 3. Show table properties
SHOW TBLPROPERTIES techmart_dw.orders;

-- 4. Time travel query
SELECT COUNT(*) FROM techmart_dw.orders VERSION AS OF 0;

-- 5. Optimize (compact small files)
-- OPTIMIZE techmart_dw.orders;

-- 6. Vacuum (remove old files)
-- VACUUM techmart_dw.orders RETAIN 168 HOURS;

In [ ]:
# Cleanup demo tables
print("🧹 CLEANUP")
spark.sql("DROP TABLE IF EXISTS techmart_dw.delta_internals_demo")
spark.sql("DROP TABLE IF EXISTS techmart_dw.small_files_demo")
spark.sql("DROP TABLE IF EXISTS techmart_dw.demo_deep_clone")
spark.sql("DROP TABLE IF EXISTS techmart_dw.demo_shallow_clone")
spark.sql("DROP TABLE IF EXISTS techmart_dw.merge_target")
print("✅ Demo tables cleaned up")